<a href="https://colab.research.google.com/github/tranvananhanhanh/AI_Agent/blob/main/GameASimpleAgentFramework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!!pip install litellm

# Important!!!
#
# <---- Set your 'OPENAI_API_KEY' as a secret over there with the "key" icon
#
# <---- You will also likely want to use the "folder" icon to add some files
#       for the agent to look at
#
import os
from google.colab import userdata

api_key = userdata.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = api_key


In [16]:
import os
import json
import time
import traceback
from litellm import completion
from dataclasses import dataclass, field
from typing import List, Callable, Dict, Any


# =========================
# PROMPT STRUCTURE
# =========================
@dataclass
class Prompt:
    messages: List[Dict] = field(default_factory=list)


# =========================
# LLM CALL (GROQ FIX)
# =========================
def generate_response(prompt: Prompt) -> str:

    response = completion(
        model="groq/llama-3.1-8b-instant",
        messages=prompt.messages,
        max_tokens=1024
    )

    content = response.choices[0].message.content
    print("RAW LLM:", content)

    # Try parse JSON
    try:
        parsed = json.loads(content)
        if "tool" in parsed:
            return json.dumps(parsed)
    except:
        pass

    # fallback
    return json.dumps({
        "tool": "terminate",
        "args": {"message": content}
    })


# =========================
# CORE CLASSES
# =========================
@dataclass(frozen=True)
class Goal:
    priority: int
    name: str
    description: str


class Action:
    def __init__(self, name, function, description, parameters, terminal=False):
        self.name = name
        self.function = function
        self.description = description
        self.parameters = parameters
        self.terminal = terminal

    def execute(self, **args):
        return self.function(**args)


class ActionRegistry:
    def __init__(self):
        self.actions = {}

    def register(self, action: Action):
        self.actions[action.name] = action

    def get_action(self, name):
        return self.actions.get(name)

    def get_actions(self):
        return list(self.actions.values())


# =========================
# MEMORY
# =========================
class Memory:
    def __init__(self):
        self.items = []

    def add_memory(self, m):
        self.items.append(m)

    def get_memories(self, limit=None):
        return self.items[-limit:] if limit else self.items


# =========================
# ENVIRONMENT (SAFE)
# =========================
class Environment:
    def execute_action(self, action: Action, args: dict):

        if action is None:
            return {"tool_executed": False, "error": "Invalid tool"}

        # sanitize args
        expected = action.parameters.get("properties", {}).keys()
        clean_args = {k: v for k, v in args.items() if k in expected}

        # check required
        required = action.parameters.get("required", [])
        for r in required:
            if r not in clean_args:
                return {"tool_executed": False, "error": f"Missing {r}"}

        try:
            result = action.execute(**clean_args)
            return {
                "tool_executed": True,
                "result": result,
                "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S")
            }
        except Exception as e:
            return {
                "tool_executed": False,
                "error": str(e),
                "traceback": traceback.format_exc()
            }


# =========================
# AGENT LANGUAGE (JSON MODE)
# =========================
class AgentLanguage:

    def construct_prompt(self, actions, goals, memory):

        tool_desc = "\n".join([
            f"{a.name}: {a.description} | args: {a.parameters}"
            for a in actions
        ])

        system = f"""
You are an AI agent.

You MUST respond ONLY in JSON:

{{
  "tool": "<tool_name>",
  "args": {{...}}
}}

TOOLS:
{tool_desc}

RULES:
- Use EXACT parameter names
- DO NOT invent fields
- If file not found -> try another action
"""

        messages = [{"role": "system", "content": system}]

        for g in goals:
            messages.append({"role": "system", "content": g.description})

        for m in memory.get_memories():
            role = "assistant" if m["type"] == "assistant" else "user"
            messages.append({"role": role, "content": m["content"]})

        return Prompt(messages)

    def parse_response(self, response):
        try:
            return json.loads(response)
        except:
            return {"tool": "terminate", "args": {"message": response}}


# =========================
# AGENT
# =========================
class Agent:
    def __init__(self, goals, language, registry, llm, env):
        self.goals = goals
        self.language = language
        self.registry = registry
        self.llm = llm
        self.env = env

    def run(self, user_input, max_iter=20):

        memory = Memory()
        memory.add_memory({"type": "user", "content": user_input})

        for _ in range(max_iter):
            print("\n--- Agent thinking ---")

            prompt = self.language.construct_prompt(
                self.registry.get_actions(),
                self.goals,
                memory
            )

            response = self.llm(prompt)
            print("Decision:", response)

            invocation = self.language.parse_response(response)
            action = self.registry.get_action(invocation["tool"])

            result = self.env.execute_action(action, invocation.get("args", {}))
            print("Result:", result)

            memory.add_memory({"type": "assistant", "content": response})
            memory.add_memory({"type": "user", "content": json.dumps(result)})

            if action and action.terminal:
                break

        return memory


# =========================
# TOOLS
# =========================
def list_project_files():
    return os.listdir(".")


def read_project_file(name: str):
    with open(name, "r") as f:
        return f.read()


def write_file(name: str, content: str):
    with open(name, "w") as f:
        f.write(content)
    return "written"


# =========================
# SETUP
# =========================
goals = [
    Goal(1, "Analyze project", "Read project files"),
    Goal(1, "Create README", "If README does not exist, create it"),
]

registry = ActionRegistry()

registry.register(Action(
    "list_project_files",
    list_project_files,
    "List files",
    {"type": "object", "properties": {}}
))

registry.register(Action(
    "read_project_file",
    read_project_file,
    "Read file",
    {
        "type": "object",
        "properties": {"name": {"type": "string"}},
        "required": ["name"]
    }
))

registry.register(Action(
    "write_file",
    write_file,
    "Write file",
    {
        "type": "object",
        "properties": {
            "name": {"type": "string"},
            "content": {"type": "string"}
        },
        "required": ["name", "content"]
    }
))

registry.register(Action(
    "terminate",
    lambda message="Done": message,
    "Terminate",
    {
        "type": "object",
        "properties": {"message": {"type": "string"}}
    },
    terminal=True
))

agent = Agent(
    goals,
    AgentLanguage(),
    registry,
    generate_response,
    Environment()
)


# =========================
# RUN
# =========================
memory = agent.run("Write a README for this project")

print("\nFINAL:")
for m in memory.get_memories():
    print(m)



--- Agent thinking ---
RAW LLM: {
  "tool": "write_file",
  "args": {
    "name": "README.md",
    "content": "# Project README\nThis project is a template for [project name].\n"
  }
}
Decision: {"tool": "write_file", "args": {"name": "README.md", "content": "# Project README\nThis project is a template for [project name].\n"}}
Result: {'tool_executed': True, 'result': 'written', 'timestamp': '2026-03-19T15:11:06'}

--- Agent thinking ---
RAW LLM: {
  "tool": "read_project_file",
  "args": {
    "name": "README.md"
  }
}
Decision: {"tool": "read_project_file", "args": {"name": "README.md"}}
Result: {'tool_executed': True, 'result': '# Project README\nThis project is a template for [project name].\n', 'timestamp': '2026-03-19T15:11:06'}

--- Agent thinking ---
RAW LLM: {"tool": "list_project_files", "args": {}}
Decision: {"tool": "list_project_files", "args": {}}
Result: {'tool_executed': True, 'result': ['.config', 'sample_data.txt', 'README.md', 'sample_data'], 'timestamp': '2026-03-

In [17]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'sample_data.txt', 'README.md', 'sample_data']
